In [ ]:
import os
import tarfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
# ==========================================
# 1. DOWNLOAD & LOAD DATASET (WITH FALLBACK)
# ==========================================
IMG_SIZE = 160
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Download official flower_photos archive via wget
if not os.path.exists("flower_photos"):
    !wget -q http://download.tensorflow.org/example_images/flower_photos.tgz
    if os.path.exists("flower_photos.tgz") and os.path.getsize("flower_photos.tgz") > 1000000:
        with tarfile.open("flower_photos.tgz", "r:gz") as tar:
            tar.extractall()
        print("Downloaded and extracted flower_photos successfully!")

if os.path.exists("flower_photos"):
    data_dir = "flower_photos"
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="training", seed=42,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
    )
    val_test_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="validation", seed=42,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
    )
    class_names = train_ds.class_names
    val_count = tf.data.experimental.cardinality(val_test_ds).numpy()
    val_ds = val_test_ds.take(val_count // 2)
    test_ds = val_test_ds.skip(val_count // 2)
else:
    # Built-in fallback if external URLs are restricted on your network
    print("External URL blocked; using built-in CIFAR-10 dataset (5 classes subset)...")
    (x_tr, y_tr), (x_te, y_te) = tf.keras.datasets.cifar10.load_data()
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer']
    mask_tr = (y_tr.flatten() < 5)
    mask_te = (y_te.flatten() < 5)
    x_tr, y_tr = x_tr[mask_tr][:3000], y_tr[mask_tr][:3000]
    x_te, y_te = x_te[mask_te][:800], y_te[mask_te][:800]

    def resize_ds(x, y):
        ds = tf.data.Dataset.from_tensor_slices((x, y))
        return ds.map(lambda img, lbl: (tf.image.resize(img, (IMG_SIZE, IMG_SIZE)), lbl)).batch(BATCH_SIZE)

    train_ds = resize_ds(x_tr[:2400], y_tr[:2400])
    val_ds = resize_ds(x_tr[2400:], y_tr[2400:])
    test_ds = resize_ds(x_te, y_te)

num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")


In [ ]:
# ==========================================
# 2. PREPROCESSING & PIPELINE OPTIMIZATION
# ==========================================
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

def format_batch(images, labels):
    return preprocess_input(images), labels

train_batches = train_ds.map(format_batch, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_batches = val_ds.map(format_batch, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_batches = test_ds.map(format_batch, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


In [ ]:
# ==========================================
# 3. BUILD MOBILENETV2 MODEL (FROZEN BASE)
# ==========================================
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze all 154 base layers for Phase 1

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.RandomFlip('horizontal')(inputs)
x = tf.keras.layers.RandomRotation(0.2)(x)
x = tf.keras.layers.RandomZoom(0.1)(x)
x = base_model(x, training=False)  # Keep BatchNorm in inference mode
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

In [ ]:
# ==========================================
# 4. PHASE 1: TRANSFER LEARNING (HEAD ONLY)
# ==========================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- PHASE 1: Training Custom Classification Head ---")
initial_epochs = 6
history_1 = model.fit(train_batches, epochs=initial_epochs, validation_data=val_batches)

In [ ]:
# ==========================================
# 5. PHASE 2: FINE-TUNING TOP BASE LAYERS
# ==========================================
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False  # Freeze bottom 100 layers; unfreeze top 54 layers

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # 100x lower LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- PHASE 2: Fine-Tuning Top MobileNetV2 Layers ---")
fine_tune_epochs = 6
total_epochs = initial_epochs + fine_tune_epochs

history_2 = model.fit(
    train_batches,
    epochs=total_epochs,
    initial_epoch=history_1.epoch[-1] + 1,
    validation_data=val_batches
)

In [ ]:




# ==========================================
# 6. EVALUATE ON UNSEEN TEST SET
# ==========================================
test_loss, test_acc = model.evaluate(test_batches, verbose=0)
print(f"\nFinal Unseen Test Accuracy: {test_acc * 100:.2f}% | Test Loss: {test_loss:.4f}")

# ==========================================
# 7. PLOT ACCURACY & LOSS CURVES
# ==========================================
acc = history_1.history['accuracy'] + history_2.history['accuracy']
val_acc = history_1.history['val_accuracy'] + history_2.history['val_accuracy']
loss = history_1.history['loss'] + history_2.history['loss']
val_loss = history_1.history['val_loss'] + history_2.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.axvline(x=initial_epochs - 1, color='gray', linestyle='--', label='Start Fine-Tuning')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.axvline(x=initial_epochs - 1, color='gray', linestyle='--', label='Start Fine-Tuning')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.tight_layout()
plt.show()

# ==========================================
# 8. VISUALIZE SAMPLE TEST PREDICTIONS
# ==========================================
image_batch, label_batch = next(iter(test_batches))
predictions = model.predict(image_batch, verbose=0)

plt.figure(figsize=(12, 6))
for i in range(8):
    ax = plt.subplot(2, 4, i + 1)
    display_img = (image_batch[i].numpy() + 1.0) / 2.0
    plt.imshow(np.clip(display_img, 0, 1))
    pred_idx = np.argmax(predictions[i])
    true_idx = int(np.squeeze(label_batch[i].numpy()))
    confidence = np.max(predictions[i]) * 100
    color = 'green' if pred_idx == true_idx else 'red'
    plt.title(f"Pred: {class_names[pred_idx]} ({confidence:.1f}%)\nTrue: {class_names[true_idx]}", color=color, fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.show()


/tmp/ipykernel_2252/1011514390.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Downloaded and extracted flower_photos successfully!
Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Classes (5): ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

--- PHASE 1: Training Custom Classification Head ---
Epoch 1/6
92/92 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - accuracy: 0.6396 - loss: 0.9647 - val_accuracy: 0.8153 - val_loss: 0.5140
Epoch 2/6
92/92 ━━━━━━━━━━━━━━━━━━━━ 117s 987ms/step - accuracy: 0.8164 - loss: 0.5145 - val_accuracy: 0.8636 - val_loss: 0.3791
Epoch 3/6
92/92 ━━━━━━━━━━━━━━━━━━━━ 140s 981ms/step - accuracy: 0.8512 - loss: 0.4138 - val_accuracy: 0.8466 - val_loss: 0.4232
Epoch 4/6
92/92 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.8624 - loss: 0.3761 - val_accuracy: 0.8722 - val_loss: 0.3547
Epoch 5/6
92/92 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.8747 - loss: 0.3509 - val_accuracy: 0.8523 - val_loss: 0.4